# ExplaineR in R — Explain a breast cytology classifier
**Ramtin Zargari Marandi, PhD**

Train a random forest, check its held-out performance, and use ExplaineR to see how recorded cell characteristics contribute to individual predictions. This example uses the Wisconsin breast cytology data distributed with **mlbench**.

[ExplaineR on CRAN](https://cran.r-project.org/package=explainer) · [Source code](https://github.com/PERSIMUNE/explainer) · [Full tutorial](https://persimune.github.io/explainer/articles/explainer_tutorial_binary_classification.html)


## 1. Set up the R environment
Use a CPU session with Internet enabled to install any missing packages.


In [ ]:
repo <- "https://cloud.r-project.org"
if (Sys.info()[["sysname"]] == "Linux" && file.exists("/etc/os-release")) {
  release <- readLines("/etc/os-release")
  distro <- sub('^VERSION_CODENAME=', '', grep('^VERSION_CODENAME=', release, value = TRUE))
  if (length(distro) == 1 && distro %in% c("noble", "jammy", "trixie")) {
    repo <- paste0("https://packagemanager.posit.co/cran/__linux__/", distro, "/latest")
  }
}
packages <- c("explainer", "mlbench", "mlr3", "mlr3learners", "ranger", "iml", "forcats")
missing <- packages[!vapply(packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) install.packages(missing, repos = repo, Ncpus = 2L)
stopifnot(all(vapply(packages, requireNamespace, logical(1), quietly = TRUE)))
library(explainer)
library(mlr3learners)
packageVersion("explainer")


## 2. Prepare data and train a classifier
Drop the sample ID and omit rows with missing values for this demonstration. The nine original predictors describe cell characteristics, such as clump thickness and uniformity of cell size. The target is the recorded benign or malignant class. Use a stratified split and keep the test observations separate from model training.


In [ ]:
RNGkind("L'Ecuyer-CMRG")
seed <- 246L
set.seed(seed)
data("BreastCancer", package = "mlbench")
samples <- na.omit(BreastCancer[, setdiff(names(BreastCancer), "Id")])
task <- mlr3::TaskClassif$new("breast_cytology", samples, target = "Class", positive = "malignant")
task$set_col_roles("Class", add_to = "stratum")
splits <- mlr3::partition(task, ratio = 0.75)
learner <- mlr3::lrn("classif.ranger", predict_type = "prob", num.threads = 2L)
learner$train(task, splits$train)
data.frame(complete_samples = nrow(samples), predictors = length(task$feature_names),
           development = length(splits$train), test = length(splits$test))


## 3. Evaluate held-out predictions
ROC AUC measures discrimination across thresholds. Balanced accuracy averages sensitivity and specificity at the classifier’s decision threshold. Inspect the confusion matrix to see which errors the model made.


In [ ]:
prediction <- learner$predict(task, splits$test)
prediction$score(mlr3::msrs(c("classif.auc", "classif.bacc")))
cm <- explainer::eCM_plot(task = task, trained_model = learner, splits = splits)
print(cm[[2]])


## 4. Explain predictions with SHAP
ExplaineR estimates feature contributions for a sample of held-out observations. Positive contributions increase the model’s predicted probability of the malignant class relative to its baseline. Colour represents the scaled feature value. The marker shape shows whether the prediction was correct. Use the plot to compare model behaviour across samples and examine errors.


In [ ]:
shap <- explainer::eSHAP_plot(task = task, trained_model = learner, splits = splits,
                            sample.size = 30, seed = seed, subset = 0.35)
shap[[1]]


The static view below uses the same ExplaineR estimates and remains readable in notebook previews. Features are ordered by mean absolute contribution.


In [ ]:
library(ggplot2)
shap_data <- as.data.frame(shap[[3]])
shap_data$feature <- reorder(shap_data$feature, abs(shap_data$Phi), FUN = mean)
ggplot(shap_data, aes(Phi, feature, colour = f_val, shape = correct_prediction)) +
  geom_vline(xintercept = 0, linetype = "dashed", colour = "grey65") +
  geom_point(position = position_jitter(width = 0, height = 0.16, seed = seed), alpha = 0.7) +
  scale_colour_viridis_c() +
  labs(title = "ExplaineR SHAP contributions on the test set",
       x = "Contribution to predicted probability", y = NULL,
       colour = "Scaled value", shape = "Prediction") +
  theme_minimal(base_size = 13)


## 5. Adapt the example
Replace the example data with your own dataset, set the target and positive class, and choose a split that matches your research design. The full ExplaineR tutorial also covers decision curves and subgroup analysis.

**Citation** · Zargari Marandi et al., *ExplaineR*, Bioinformatics Advances. [Methods paper](https://doi.org/10.1093/bioadv/vbae049)
